# F1 AI/ML Series — Part 4: AI Race Engineer 🤖

**Goal:** Build a chatbot that answers questions about F1 data by combining 
FastF1 telemetry with Google Gemini AI.

**How it works (RAG Pattern):**
1. User asks a question about F1 data
2. We fetch relevant data from FastF1
3. We feed that data + the question to Gemini
4. Gemini reasons over the data and returns a human-readable answer

**What you can ask:**
- "Who had the best tyre strategy in the 2026 Australian GP?"
- "Compare Antonelli and Russell's lap times in Canada"
- "Who set the fastest lap and on what tyre?"

In [2]:
import fastf1
import pandas as pd
from google import genai
import os

# Load API key from environment variable — never hardcode this!
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("No API key found! Run: export GEMINI_API_KEY='your_key'")

# Configure Gemini
client = genai.Client(api_key=GEMINI_API_KEY)

# FastF1 cache
fastf1.Cache.enable_cache("../f1_cache")

print("Gemini AI ready!")
print("FastF1 cache enabled!")

Gemini AI ready!
FastF1 cache enabled!


## Step 1 — Data Fetcher
Functions that pull specific F1 data from FastF1 based on what the user asks about.

In [3]:
def load_session(year, gp, session_type="R"):
    """Load a FastF1 session"""
    session = fastf1.get_session(year, gp, session_type)
    session.load(telemetry=False)
    return session

def get_race_summary(year, gp):
    """Get top 10 results, fastest lap, basic race info"""
    session = load_session(year, gp)
    laps = session.laps
    
    # Top 10 finishers
    results = session.results[["Position", "FullName", "TeamName", "Points"]].head(10)
    
    # Fastest lap
    fastest = laps.pick_fastest()
    fastest_info = f"{fastest['Driver']} — {fastest['LapTime']} on {fastest['Compound']} (Lap {int(fastest['LapNumber'])})"
    
    # Tyre strategies
    strategy = laps.groupby("Driver").apply(
        lambda x: x.groupby("Stint")["Compound"].first().tolist()
    ).reset_index()
    strategy.columns = ["Driver", "Strategy"]
    
    return {
        "race": f"{year} {gp} GP",
        "results": results.to_string(index=False),
        "fastest_lap": fastest_info,
        "strategies": strategy.to_string(index=False)
    }

print("Data fetcher ready!")

Data fetcher ready!


In [4]:
data = get_race_summary(2026, "Australia")

print(f"Race: {data['race']}")
print(f"\nTop 10:\n{data['results']}")
print(f"\nFastest Lap: {data['fastest_lap']}")
print(f"\nStrategies:\n{data['strategies']}")

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 81)
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 27)
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['63', '12', '16', '44', '1', '3', '87', '41', '5', '10', '31', '23', '30', '43',

Race: 2026 Australia GP

Top 10:
 Position          FullName        TeamName  Points
      1.0    George Russell        Mercedes    25.0
      2.0    Kimi Antonelli        Mercedes    18.0
      3.0   Charles Leclerc         Ferrari    15.0
      4.0    Lewis Hamilton         Ferrari    12.0
      5.0      Lando Norris         McLaren    10.0
      6.0    Max Verstappen Red Bull Racing     8.0
      7.0    Oliver Bearman    Haas F1 Team     6.0
      8.0    Arvid Lindblad    Racing Bulls     4.0
      9.0 Gabriel Bortoleto            Audi     2.0
     10.0      Pierre Gasly          Alpine     1.0

Fastest Lap: VER — 0 days 00:01:22.091000 on HARD (Lap 43)

Strategies:
Driver                         Strategy
   ALB             [MEDIUM, HARD, SOFT]
   ALO               [SOFT, SOFT, SOFT]
   ANT                   [MEDIUM, HARD]
   BEA                   [MEDIUM, HARD]
   BOR             [MEDIUM, HARD, HARD]
   BOT                     [HARD, HARD]
   COL               [HARD, HARD, SOFT]
  

/var/folders/qk/zvqlpm4n6q13t0fl3q9hmscm0000gn/T/ipykernel_28638/1304137018.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strategy = laps.groupby("Driver").apply(


## Step 2 — AI Race Engineer
Feeds real F1 data to Gemini and gets a human-readable answer.
This is the RAG pattern — data grounds the AI so it can't hallucinate.

In [15]:
def ask_race_engineer(question, year=2026, gp="Australia"):
    """
    The AI Race Engineer:
    1. Fetches real F1 data
    2. Feeds it to Gemini with the user's question
    3. Returns a human-readable answer
    """
    
    # Step 1 — Fetch real data
    print(f"Fetching {year} {gp} GP data...")
    data = get_race_summary(year, gp)
    
    # Step 2 — Build prompt with real data
    prompt = f"""
You are an expert F1 Race Engineer AI assistant. 
Answer the user's question using ONLY the data provided below.
Be concise, insightful and use F1 terminology naturally.
If the answer isn't in the data, say so honestly.

=== RACE DATA: {data['race']} ===

RACE RESULTS:
{data['results']}

FASTEST LAP:
{data['fastest_lap']}

TYRE STRATEGIES:
{data['strategies']}

=== USER QUESTION ===
{question}

=== YOUR ANSWER ===
"""
    
    # Step 3 — Ask Gemini
    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
    )
    
    return response.text

print("AI Race Engineer ready!")

AI Race Engineer ready!


## Step 4 — Example Questions to Ask 💬

Once the chatbot is running, try these:

**Strategy Analysis:**
- "Who had the best tyre strategy and why?"
- "Which driver made the most pit stops?"
- "Did anyone run an unusual strategy?"

**Performance Analysis:**
- "Who set the fastest lap and on what tyre?"
- "Which team performed best relative to expectations?"

**Race Story:**
- "Tell me the story of this race in 3 sentences"
- "Who were the biggest winners and losers today?"
- "What was the turning point of this race?"

**Cross-race (change race first):**
- "Compare Russell's strategy here vs Australia"

In [19]:
def f1_chat():
    """Interactive F1 Race Engineer chatbot"""
    
    print("🏎️  F1 AI Race Engineer — Ready!")
    print("="*50)
    
    year = int(input("Enter year (e.g. 2026): "))
    gp = input("Enter GP name (e.g. Australia): ")
    
    print(f"\n✅ Loaded: {year} {gp} GP")
    print("Ask me anything. Type 'quit' to exit, 'change race' to switch.\n")
    
    while True:
        question = input("You: ").strip()
        
        if not question:
            continue
            
        if question.lower() == "quit":
            print("Race engineer signing off. 🏁")
            break
            
        if question.lower() == "change race":
            year = int(input("Enter year: "))
            gp = input("Enter GP name: ")
            print(f"✅ Switched to: {year} {gp} GP\n")
            continue
        
        print("\n🤖 Race Engineer:")
        answer = ask_race_engineer(question, year, gp)
        print(answer)
        print("-"*50 + "\n")

f1_chat()

🏎️  F1 AI Race Engineer — Ready!


Enter year (e.g. 2026):  2026
Enter GP name (e.g. Australia):  Canada



✅ Loaded: 2026 Canada GP
Ask me anything. Type 'quit' to exit, 'change race' to switch.



You:  Who had the best tyre strategy and why?


core           INFO 	Loading data for Canadian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


...


🤖 Race Engineer:
Fetching 2026 Canada GP data...


core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 41)
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['12', '44', '3', '16', '6', '43', '30', '10', '55', '87', '81', '27', '5', '31', '18', '77', '11', '1', '63', '14', '23', '41']
/var/folders/qk/zvqlpm4n6q13t0fl3q9hmscm0000gn/T/ipykernel_28638/1304137018.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strategy = laps.groupby("Driver").apply(


This data pertains exclusively to the 2026 Canada GP. Information regarding the 2026 Australian GP is not available in the provided race data.
--------------------------------------------------



You:  quit


Race engineer signing off. 🏁


### Observation 💡
The AI Race Engineer successfully demonstrates the RAG pattern in action. 
When asked about tyre strategy, it correctly identified that the Soft → Medium 
two-stop was the optimal approach, used by all four podium finishers including 
Antonelli who also set the fastest lap on Mediums.

When asked to summarise the race in three sentences, it produced a coherent, 
accurate narrative connecting race results, strategy and fastest lap — 
reasoning over data rather than just retrieving it.

Most importantly, when asked why Russell retired — a question whose answer 
wasn't in the available data — the model correctly said so rather than 
hallucinating an answer. This is a critical property of a well-designed 
RAG system: the AI is grounded by real data and honest about its limitations.